# PyStrata workflow

Import the package directly from this repository's `src` directory.

In [1]:
import sys
from pathlib import Path

# Find the repository root whether Jupyter starts in the root or tests/.
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "pystrata").is_dir()
)

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pystrata
from pystrata.output import KappaOutput

print(f"Using pystrata from: {Path(pystrata.__file__).resolve()}")

Using pystrata from: C:\Users\jimxi\GitHub\pystrata\src\pystrata\__init__.py


In [2]:
import numpy as np
import pyrvt
import pandas as pd

In [3]:
Site_Profile_df = pd.read_excel('data/Meloland_Soil_Profile.xlsx',
                               sheet_name = 'Scaled_Dmin_to_Kappa_0.036')

Layers = []
D_min = []
Vs = []
Thickness = []
Profile_Depth = []
mrd_strains = np.logspace(-6,0,num=20)
ModReduc_data = {}
Damping_data = {}
max_freqs = Site_Profile_df['Max Freq']
wave_fracs = Site_Profile_df['Wave Fraction']

In [4]:

for i, (_, row) in enumerate(Site_Profile_df.iterrows()):

    soil_type = pystrata.site.DarendeliSoilType(unit_wt = row['Unit Weight (kN/m3)'],
                                                plas_index=row['PI'],
                                                ocr=1,
                                                stress_mean=row['Stress (kPa)'],
                                                strains = mrd_strains,
                                                damping_min = row['Scaled_D_min (%)']/100)
    
    ModReduc_data[f"Layer {i+1}"] = soil_type.mod_reduc.values
    Damping_data[f"Layer {i+1}"] = soil_type.damping.values * 100
    
    Layers.append(pystrata.site.Layer(soil_type,row['Thickness (m)'],row['Velocity (m/s)']))

Layers.append(
            pystrata.site.Layer(
                pystrata.site.SoilType(
                    'Reference Rock',
                    25.9,
                    None,
                    0.01
                ),
                0,
                3500
            )
        )

In [5]:
Site_profile = pystrata.site.Profile(Layers)
discretized_Site_profile = Site_profile.auto_discretize(max_freq = max_freqs,wave_frac=wave_fracs)

In [6]:

# Calculation Loop
outputs_freqs = np.logspace(np.log10(0.01),np.log10(100),1000)
RS_freqs = np.logspace(np.log10(0.05),np.log10(100),1000)

output = pystrata.output.OutputCollection(
    [
        pystrata.output.AccelTransferFunctionOutput(
            # Frequency
            outputs_freqs,
            # Location in (denominator),
            pystrata.output.OutputLocation("outcrop", index=-1),
            # Location out (numerator)
            pystrata.output.OutputLocation("outcrop", index=0)
        ),
        pystrata.output.FourierAmplitudeSpectrumOutput(
            outputs_freqs,
            pystrata.output.OutputLocation("outcrop", index=0),
            None #type:ignore
        ),
        pystrata.output.AriasIntensityTSOutput(
            pystrata.output.OutputLocation("outcrop", index=0)
        ),
        pystrata.output.ResponseSpectrumOutput(
            # Frequency
            RS_freqs,
            # Location of the output
            pystrata.output.OutputLocation("outcrop", index=0),
            # Damping
            0.05,
        ),
        pystrata.output.ResponseSpectrumRatioOutput(
            # Frequency
            RS_freqs,
            # Location of the output
            pystrata.output.OutputLocation("outcrop", index=-1),
            pystrata.output.OutputLocation("outcrop", index=0),
            # Damping
            0.05,
        ),  
        pystrata.output.MaxStrainProfile()
    ] 
    )


motion = pystrata.motion.TimeSeriesMotion.load_at2_file(
    'data/NIS090.AT2'
)
eql_calc = pystrata.propagation.EquivalentLinearCalculator(strain_limit = 0.5)
fd_eql_calc = pystrata.propagation.FrequencyDependentEqlCalculator(strain_limit = 0.5,method="ko:30")
le_calc = pystrata.propagation.LinearElasticCalculator()




p = discretized_Site_profile.copy()

eql_calc(motion, #type:ignore
        p,
        p.location("outcrop", index=-1))
            
output(eql_calc,
        name = f"test",)

In [7]:
help(output[2])

Help on AriasIntensityTSOutput in module pystrata.output object:

class AriasIntensityTSOutput(AccelerationTSOutput)
 |  AriasIntensityTSOutput(location)
 |
 |  Method resolution order:
 |      AriasIntensityTSOutput
 |      AccelerationTSOutput
 |      TimeSeriesOutput
 |      LocationBasedOutput
 |      Output
 |      builtins.object
 |
 |  Data and other attributes defined here:
 |
 |  ylabel = 'Arias Intensity (m/s)'
 |
 |  ----------------------------------------------------------------------
 |  Methods inherited from TimeSeriesOutput:
 |
 |  __call__(self, calc, name=None)
 |      Call self as a function.
 |
 |  __init__(self, location)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  to_dataframe(self)
 |
 |  ----------------------------------------------------------------------
 |  Readonly properties inherited from TimeSeriesOutput:
 |
 |  times
 |
 |  ----------------------------------------------------------------------
 |  Data and other attrib

In [9]:
print(output[2].values)

[1.67946396e-05 3.41295355e-05 5.18492366e-05 ... 2.06568326e+00
 2.06569761e+00 2.06571290e+00]
